# Lab 2 : Deep Learning

Implémentation d'un réseau de neurones multicouche et de la backpropagation à la main, sans librairie de deep learning. On utilise seulement `random`, `math` et `csv`.

Un réseau est une liste de couches, une couche est une liste de neurones, et un neurone est un dictionnaire avec ses poids (le biais est le dernier poids).

## 1. Initialisation du réseau

Questions 1 à 3. Les poids sont tirés uniformément dans [0, 1[ et arrondis à une décimale. On ne met pas la graine dans la fonction, on la fixe avant chaque test.

In [1]:
import random
import math
import csv


def initialize_network(n_inputs, n_hidden, n_outputs):
    network = []
    variable = [n_inputs, n_hidden, n_outputs]
    for i in range(1, len(variable)):
        layer = []
        for j in range(variable[i]):
            neuro = {"weights": [round(random.random(), 1) for _ in range(variable[i-1] + 1)]}
            layer.append(neuro)
        network.append(layer)
    return network


def write(network):
    for i in range(len(network)):
        if i != len(network) - 1:
            print("HIDDEN LAYER", i + 1)
        else:
            print("OUTPUT")
        for j in range(len(network[i])):
            for keys, values in network[i][j].items():
                print("   ", keys, ":", values)
    print()


random.seed(1)
net = initialize_network(2, 3, 4)
write(net)

HIDDEN LAYER 1
    weights : [0.1, 0.8, 0.8]
    weights : [0.3, 0.5, 0.4]
    weights : [0.7, 0.8, 0.1]
OUTPUT
    weights : [0.0, 0.8, 0.4, 0.8]
    weights : [0.0, 0.4, 0.7, 0.2]
    weights : [0.9, 0.9, 0.0, 0.0]
    weights : [0.5, 0.9, 0.4, 0.2]



On a bien 3 neurones cachés avec 2 + 1 = 3 poids chacun, et 4 neurones de sortie avec 3 + 1 = 4 poids chacun.

## 2. Vérification de l'initialisation

Question 4. Avec la même graine on doit retrouver exactement les mêmes poids.

In [2]:
random.seed(1)
net = initialize_network(3, 2, 2)
write(net)

random.seed(1)
net_bis = initialize_network(3, 2, 2)
print("Même réseau en relançant :", net == net_bis)

HIDDEN LAYER 1
    weights : [0.1, 0.8, 0.8, 0.3]
    weights : [0.5, 0.4, 0.7, 0.8]
OUTPUT
    weights : [0.1, 0.0, 0.8]
    weights : [0.4, 0.8, 0.0]

Même réseau en relançant : True


## 3. Activation d'un neurone

Questions 5 et 6. C'est la somme des poids fois les entrées, plus le biais.

In [3]:
def activate(weights, inputs):
    activation = 0
    for i in range(len(weights) - 1):
        activation += weights[i] * inputs[i]
    activation += weights[-1]
    return activation


print(activate([3.0, 5.0, 2], [-1, 1]))

4.0


On vérifie à la main : 3 x (-1) + 5 x 1 + 2 = 4.

## 4. Fonction de transfert

Question 7. Sigmoïde avec `math.exp` (pas de numpy).

In [4]:
def transfer(activation):
    return 1 / (1 + math.exp(-activation))


print(transfer(activate([3.0, 5.0, 2], [-1, 1])))

0.9820137900379085


## 5. Propagation avant

Questions 8 et 9. La première couche reçoit la ligne d'entrée, les suivantes reçoivent les sorties de la couche d'avant. On stocke la sortie dans chaque neurone et la fonction renvoie la sortie du réseau.

In [5]:
def forward_propagate(network, row):
    for i in range(len(network)):
        for j in range(len(network[i])):
            if i == 0:
                network[i][j]['output'] = transfer(activate(network[i][j]['weights'], row))
            else:
                network[i][j]['output'] = transfer(activate(network[i][j]['weights'], [network[i-1][k]['output'] for k in range(len(network[i-1]))]))
    return [neuron['output'] for neuron in network[-1]]


random.seed(1)
net1 = initialize_network(2, 1, 2)
print("AVANT")
write(net1)

output = forward_propagate(net1, [1, 0])
print("APRES")
write(net1)
print("Sortie du réseau :", output)

AVANT
HIDDEN LAYER 1
    weights : [0.1, 0.8, 0.8]
OUTPUT
    weights : [0.3, 0.5]
    weights : [0.4, 0.7]

APRES
HIDDEN LAYER 1
    weights : [0.1, 0.8, 0.8]
    output : 0.7109495026250039
OUTPUT
    weights : [0.3, 0.5]
    output : 0.6711265850507707
    weights : [0.4, 0.7]
    output : 0.7279763996434003

Sortie du réseau : [0.6711265850507707, 0.7279763996434003]


## 6. Dérivée de la fonction de transfert

Question 10. Pour la sigmoïde, la dérivée s'écrit avec la sortie : y(1 - y).

In [6]:
def transfer_derivative(output):
    return output * (1 - output)


print(transfer_derivative(0.5))

0.25


## 7. Backpropagation de l'erreur

Questions 11 et 12. On part de la couche de sortie et on remonte. Pour un neurone caché, on fait la somme sur tous les neurones de la couche suivante (avant on écrasait `error` à chaque tour de boucle, donc seul le dernier neurone comptait).

In [7]:
def backward_propagate_error(network, expected):
    for i in range(len(network) - 1, -1, -1):
        for j in range(len(network[i])):
            if i == len(network) - 1:
                network[i][j]['delta'] = (expected[j] - network[i][j]['output']) * transfer_derivative(network[i][j]['output'])
            else:
                error = 0
                for k in range(len(network[i+1])):
                    error += network[i+1][k]['weights'][j] * network[i+1][k]['delta']
                network[i][j]['delta'] = error * transfer_derivative(network[i][j]['output'])


print("AVANT")
write(net1)

backward_propagate_error(net1, [0, 1])
print("APRES")
write(net1)

AVANT
HIDDEN LAYER 1
    weights : [0.1, 0.8, 0.8]
    output : 0.7109495026250039
OUTPUT
    weights : [0.3, 0.5]
    output : 0.6711265850507707
    weights : [0.4, 0.7]
    output : 0.7279763996434003

APRES
HIDDEN LAYER 1
    weights : [0.1, 0.8, 0.8]
    output : 0.7109495026250039
    delta : -0.004704162927836108
OUTPUT
    weights : [0.3, 0.5]
    output : 0.6711265850507707
    delta : -0.1481281685644896
    weights : [0.4, 0.7]
    output : 0.7279763996434003
    delta : 0.053867952550112806



La sortie 0 doit valoir 0 mais vaut environ 0.67, donc son delta est négatif. La sortie 1 doit valoir 1, son delta est positif.

## 8. Mise à jour des poids

Question 13. w = w + lr x delta x entrée, et pour le biais l'entrée vaut 1.

In [8]:
def update_weights(network, row, l_rate):
    for i in range(len(network)):
        for j in range(len(network[i])):
            for k in range(len(network[i-1]) if i > 0 else len(row)):
                if i == 0:
                    network[i][j]['weights'][k] = network[i][j]['weights'][k] + l_rate * network[i][j]['delta'] * row[k]
                else:
                    network[i][j]['weights'][k] = network[i][j]['weights'][k] + l_rate * network[i][j]['delta'] * network[i-1][k]['output']
            network[i][j]['weights'][-1] = network[i][j]['weights'][-1] + l_rate * network[i][j]['delta']

## 9. Entraînement

Questions 14 et 15. J'ai ajouté un paramètre `print_every` (1 par défaut) pour ne pas afficher 500 lignes plus tard sur le Seeds Dataset.

In [9]:
def train_network(network, train, l_rate, n_epoch, n_outputs, print_every=1):
    for epoch in range(n_epoch):
        sum_error = 0
        for row in train:
            inputs = row[:-1]
            outputs = forward_propagate(network, inputs)

            expected = [0] * n_outputs
            expected[int(row[-1])] = 1

            sum_error += sum((expected[k] - outputs[k])**2 for k in range(n_outputs))
            backward_propagate_error(network, expected)
            update_weights(network, inputs, l_rate)
        if epoch % print_every == 0 or epoch == n_epoch - 1:
            print(f"epoch={epoch}, lrate={l_rate}, error={sum_error:.3f}")

In [10]:
dataset = [[2.7810836, 2.550537003, 0],
           [1.465489372, 2.362125076, 0],
           [3.396561688, 4.400293529, 0],
           [1.38807019, 1.850220317, 0],
           [3.06407232, 3.005305973, 0],
           [7.627531214, 2.759262235, 1],
           [5.332441248, 2.088626775, 1],
           [6.922596716, 1.77106367, 1],
           [8.675418651, -0.242068655, 1],
           [7.673756466, 3.508563011, 1]]

n_inputs = len(dataset[0]) - 1
n_outputs = len(set(row[-1] for row in dataset))

random.seed(1)
network = initialize_network(n_inputs, 2, n_outputs)
train_network(network, dataset, 0.5, 40, n_outputs)

epoch=0, lrate=0.5, error=6.334
epoch=1, lrate=0.5, error=5.477
epoch=2, lrate=0.5, error=4.991
epoch=3, lrate=0.5, error=4.535
epoch=4, lrate=0.5, error=4.196
epoch=5, lrate=0.5, error=3.857
epoch=6, lrate=0.5, error=3.521
epoch=7, lrate=0.5, error=3.201
epoch=8, lrate=0.5, error=2.901
epoch=9, lrate=0.5, error=2.626
epoch=10, lrate=0.5, error=2.375
epoch=11, lrate=0.5, error=2.149
epoch=12, lrate=0.5, error=1.947
epoch=13, lrate=0.5, error=1.767
epoch=14, lrate=0.5, error=1.606
epoch=15, lrate=0.5, error=1.463
epoch=16, lrate=0.5, error=1.335
epoch=17, lrate=0.5, error=1.220
epoch=18, lrate=0.5, error=1.118
epoch=19, lrate=0.5, error=1.026
epoch=20, lrate=0.5, error=0.943
epoch=21, lrate=0.5, error=0.868
epoch=22, lrate=0.5, error=0.801
epoch=23, lrate=0.5, error=0.740
epoch=24, lrate=0.5, error=0.685
epoch=25, lrate=0.5, error=0.636
epoch=26, lrate=0.5, error=0.592
epoch=27, lrate=0.5, error=0.552
epoch=28, lrate=0.5, error=0.515
epoch=29, lrate=0.5, error=0.483
epoch=30, lrate=0.5,

L'erreur diminue bien au fil des epochs.

## 10. Prédiction

Questions 16 et 17. La classe prédite est l'indice de la plus grande sortie.

In [11]:
def predict(network, row):
    outputs = forward_propagate(network, row[:-1])
    return outputs.index(max(outputs))


correct = 0
for row in dataset:
    prediction = predict(network, row)
    correct += prediction == row[-1]
    print(f"Attendu={row[-1]}, Prédit={prediction}")
print(f"\nAccuracy : {correct}/{len(dataset)}")

Attendu=0, Prédit=0
Attendu=0, Prédit=0
Attendu=0, Prédit=0
Attendu=0, Prédit=0
Attendu=0, Prédit=0
Attendu=1, Prédit=1
Attendu=1, Prédit=1
Attendu=1, Prédit=1
Attendu=1, Prédit=1
Attendu=1, Prédit=1

Accuracy : 10/10


## 11. Bonus : généralisation

### Question 18 : choix de l'activation par couche

On passe une liste `activations` avec une activation par couche, par exemple `['tanh', 'softmax']`. Le softmax se calcule sur toute la couche d'un coup, donc on calcule d'abord toutes les pré-activations de la couche puis on applique la fonction. Pour la dérivée, on l'exprime toujours avec la sortie du neurone, comme pour la sigmoïde.

In [12]:
def sigmoid(a):
    if a < -500:  # évite un overflow de math.exp
        return 0.0
    return 1 / (1 + math.exp(-a))


def softmax(values):
    m = max(values)
    exps = [math.exp(v - m) for v in values]
    total = sum(exps)
    return [e / total for e in exps]


def transfer_g(activation, name):
    if name == 'linear':
        return activation
    if name == 'relu':
        return max(0.0, activation)
    if name == 'tanh':
        return math.tanh(activation)
    return sigmoid(activation)


def transfer_derivative_g(output, name):
    if name == 'linear':
        return 1.0
    if name == 'relu':
        return 1.0 if output > 0 else 0.0
    if name == 'tanh':
        return 1 - output**2
    return output * (1 - output)


def forward_propagate_g(network, row, activations=None):
    if activations is None:
        activations = ['sigmoid'] * len(network)
    inputs = row
    for i in range(len(network)):
        pre = [activate(neuron['weights'], inputs) for neuron in network[i]]
        if activations[i] == 'softmax':
            outs = softmax(pre)
        else:
            outs = [transfer_g(a, activations[i]) for a in pre]
        for j in range(len(network[i])):
            network[i][j]['output'] = outs[j]
        inputs = outs
    return inputs

Petite vérif : avec que des sigmoïdes on doit retrouver exactement la propagation de la partie 5.

In [13]:
random.seed(1)
a = initialize_network(2, 1, 2)
random.seed(1)
b = initialize_network(2, 1, 2)
print(forward_propagate(a, [1, 0]))
print(forward_propagate_g(b, [1, 0], ['sigmoid', 'sigmoid']))
print(softmax([1.0, 2.0, 3.0]), sum(softmax([1.0, 2.0, 3.0])))

[0.6711265850507707, 0.7279763996434003]
[0.6711265850507707, 0.7279763996434003]
[0.09003057317038046, 0.24472847105479764, 0.6652409557748218] 0.9999999999999999


### Question 19 : choix de la loss

- `'mse'` : comme avant, delta = (attendu - sortie) x dérivée. Si la sortie est un softmax, chaque sortie dépend de toutes les pré-activations, donc on utilise la vraie dérivée du softmax.
- `'cross_entropy'` : avec une sortie softmax ou sigmoïde, le delta se simplifie. Le sujet donne ŷ - y, nous on garde notre convention de signe (w = w + lr x delta x entrée), donc ici delta = y - ŷ.

Pour comparer des datasets de tailles différentes, j'affiche la loss moyenne par exemple. `train_network_g` renvoie aussi l'historique de la loss.

In [14]:
def backward_propagate_error_g(network, expected, activations=None, loss='mse'):
    if activations is None:
        activations = ['sigmoid'] * len(network)
    last = len(network) - 1
    for i in range(last, -1, -1):
        layer = network[i]
        if i == last:
            outputs = [neuron['output'] for neuron in layer]
            if loss == 'cross_entropy':
                if activations[i] not in ('softmax', 'sigmoid'):
                    raise ValueError("cross_entropy prévu avec une sortie softmax ou sigmoid")
                for j in range(len(layer)):
                    layer[j]['delta'] = expected[j] - outputs[j]
            elif activations[i] == 'softmax':
                g = [expected[k] - outputs[k] for k in range(len(layer))]
                s = sum(g[k] * outputs[k] for k in range(len(layer)))
                for j in range(len(layer)):
                    layer[j]['delta'] = outputs[j] * (g[j] - s)
            else:
                for j in range(len(layer)):
                    layer[j]['delta'] = (expected[j] - outputs[j]) * transfer_derivative_g(outputs[j], activations[i])
        else:
            for j in range(len(layer)):
                error = 0.0
                for neuron in network[i+1]:
                    error += neuron['weights'][j] * neuron['delta']
                layer[j]['delta'] = error * transfer_derivative_g(layer[j]['output'], activations[i])


def compute_loss(expected, outputs, loss, last_activation):
    if loss == 'mse':
        return sum((expected[k] - outputs[k])**2 for k in range(len(expected)))
    eps = 1e-12
    if last_activation == 'softmax':
        return -sum(expected[k] * math.log(outputs[k] + eps) for k in range(len(expected)))
    return -sum(expected[k] * math.log(outputs[k] + eps) + (1 - expected[k]) * math.log(1 - outputs[k] + eps)
                for k in range(len(expected)))


def train_network_g(network, train, l_rate, n_epoch, n_outputs, activations=None, loss='mse',
                    task='classification', print_every=0):
    if activations is None:
        activations = ['sigmoid'] * len(network)
    history = []
    for epoch in range(n_epoch):
        sum_error = 0
        for row in train:
            inputs = row[:-1]
            outputs = forward_propagate_g(network, inputs, activations)
            if task == 'classification':
                expected = [0] * n_outputs
                expected[int(row[-1])] = 1
            else:
                expected = [row[-1]]
            sum_error += compute_loss(expected, outputs, loss, activations[-1])
            backward_propagate_error_g(network, expected, activations, loss)
            update_weights(network, inputs, l_rate)
        history.append(sum_error / len(train))
        if print_every and (epoch % print_every == 0 or epoch == n_epoch - 1):
            print(f"epoch={epoch}, lrate={l_rate}, loss={history[-1]:.4f}")
    return history


def predict_g(network, row, activations=None, task='classification'):
    outputs = forward_propagate_g(network, row[:-1], activations)
    if task == 'classification':
        return outputs.index(max(outputs))
    return outputs[0]

Vérif : avec sigmoïde + mse, `train_network_g` doit donner exactement le même réseau que `train_network` sur le dataset jouet.

In [15]:
random.seed(1)
net_a = initialize_network(2, 2, 2)
random.seed(1)
net_b = initialize_network(2, 2, 2)

train_network(net_a, dataset, 0.5, 40, 2, print_every=100)
train_network_g(net_b, dataset, 0.5, 40, 2, ['sigmoid', 'sigmoid'], 'mse')
print("Mêmes poids :", [n['weights'] for l in net_a for n in l] == [n['weights'] for l in net_b for n in l])

epoch=0, lrate=0.5, error=6.334
epoch=39, lrate=0.5, error=0.281
Mêmes poids : True


### Question 20 : comparaison de configurations

Méthode suivie pour les trois problèmes :

- séparation train / test (70 / 30) avec une graine fixe, on évalue toujours sur le test ;
- standardisation des features avec la moyenne et l'écart-type du train seulement, pour ne pas faire fuiter d'information du test ;
- d'abord une baseline très simple (classe majoritaire, ou moyenne de y en régression) pour savoir à partir de quand un score veut dire quelque chose ;
- même initialisation (seed 1) et même nombre d'epochs pour toutes les configurations d'un même problème.

In [16]:
def train_test_split(data, ratio=0.7, seed=0):
    rows = [list(r) for r in data]
    random.seed(seed)
    random.shuffle(rows)
    cut = int(len(rows) * ratio)
    return rows[:cut], rows[cut:]


def standardize(train, test):
    n = len(train[0]) - 1
    means = [sum(r[i] for r in train) / len(train) for i in range(n)]
    stds = [math.sqrt(sum((r[i] - means[i])**2 for r in train) / len(train)) for i in range(n)]
    for data in (train, test):
        for r in data:
            for i in range(n):
                r[i] = (r[i] - means[i]) / stds[i]


def accuracy(network, data, activations):
    good = sum(predict_g(network, r, activations) == r[-1] for r in data)
    return good / len(data)


def majority_baseline(train, test):
    labels = [r[-1] for r in train]
    most = max(set(labels), key=labels.count)
    return sum(r[-1] == most for r in test) / len(test)


def run_config(train, test, n_hidden, activations, loss, n_outputs, l_rate, n_epoch, task='classification'):
    random.seed(1)
    network = initialize_network(len(train[0]) - 1, n_hidden, n_outputs)
    history = train_network_g(network, train, l_rate, n_epoch, n_outputs, activations, loss, task)
    return network, history


def show_history(name, history, n_points=6):
    steps = [int(i * (len(history) - 1) / (n_points - 1)) for i in range(n_points)]
    print(f"{name:<28}" + "  ".join(f"ep{s}={history[s]:.3f}" for s in steps))

#### Problème 1 : classification binaire

Points dans le carré [-1, 1]², classe 1 si le point est dans le cercle de rayon 0.7. C'est non linéaire, donc il faut vraiment la couche cachée.

In [17]:
def make_circle(n, seed):
    random.seed(seed)
    data = []
    for _ in range(n):
        x1, x2 = random.uniform(-1, 1), random.uniform(-1, 1)
        data.append([x1, x2, 1 if x1**2 + x2**2 < 0.49 else 0])
    return data


circle = make_circle(300, seed=42)
c_train, c_test = train_test_split(circle)
standardize(c_train, c_test)
print("train :", len(c_train), " test :", len(c_test))
print("Baseline classe majoritaire (test) :", round(majority_baseline(c_train, c_test), 3))

train : 210  test : 90
Baseline classe majoritaire (test) : 0.578


In [18]:
configs_bin = [
    ("sigmoid / sigmoid / mse", ['sigmoid', 'sigmoid'], 'mse'),
    ("tanh / sigmoid / mse", ['tanh', 'sigmoid'], 'mse'),
    ("tanh / softmax / CE", ['tanh', 'softmax'], 'cross_entropy'),
    ("relu / softmax / CE", ['relu', 'softmax'], 'cross_entropy'),
]

print("Evolution de la loss moyenne (train) :")
results_bin = []
for name, acts, loss in configs_bin:
    network, history = run_config(c_train, c_test, 8, acts, loss, 2, 0.1, 200)
    show_history(name, history)
    results_bin.append((name, accuracy(network, c_train, acts), accuracy(network, c_test, acts)))

print("\nConfiguration                train acc   test acc")
for name, tr, te in results_bin:
    print(f"{name:<28} {tr:.3f}       {te:.3f}")

Evolution de la loss moyenne (train) :


sigmoid / sigmoid / mse     ep0=0.688  ep39=0.422  ep79=0.179  ep119=0.127  ep159=0.107  ep199=0.089


tanh / sigmoid / mse        ep0=0.658  ep39=0.103  ep79=0.058  ep119=0.044  ep159=0.038  ep199=0.034


tanh / softmax / CE         ep0=0.694  ep39=0.092  ep79=0.070  ep119=0.047  ep159=0.039  ep199=0.036


relu / softmax / CE         ep0=0.629  ep39=0.127  ep79=0.074  ep119=0.057  ep159=0.052  ep199=0.050

Configuration                train acc   test acc
sigmoid / sigmoid / mse      0.957       0.922
tanh / sigmoid / mse         0.990       0.933
tanh / softmax / CE          0.995       0.944
relu / softmax / CE          0.995       0.956


Toutes les configurations battent largement la baseline (0.578). Le réseau du TD (sigmoïde partout + mse) est le plus lent : à loss égale (mse), la version tanh descend à 0.10 dès l'epoch 39 alors que la version sigmoïde est encore à 0.42. La tanh est centrée en 0 et sa dérivée monte jusqu'à 1, contre 0.25 au maximum pour la sigmoïde, donc les deltas sont plus grands.

Les valeurs de mse et de cross-entropy ne se comparent pas directement entre elles. Sur le test les écarts sont petits (un point de test = 1.1 %), donc je ne conclus pas que relu est la meilleure, seulement que sigmoïde + mse converge le moins vite.

#### Problème 2 : classification à 3 classes

Trois nuages de points gaussiens qui se chevauchent un peu.

In [19]:
def make_blobs(n_per_class, seed):
    random.seed(seed)
    centers = [(0, 0), (2.5, 2.5), (0, 3.5)]
    data = []
    for label, (cx, cy) in enumerate(centers):
        for _ in range(n_per_class):
            data.append([random.gauss(cx, 1), random.gauss(cy, 1), label])
    return data


blobs = make_blobs(100, seed=7)
b_train, b_test = train_test_split(blobs)
standardize(b_train, b_test)
print("train :", len(b_train), " test :", len(b_test))
print("Baseline classe majoritaire (test) :", round(majority_baseline(b_train, b_test), 3))

train : 210  test : 90
Baseline classe majoritaire (test) : 0.278


In [20]:
configs_multi = [
    ("sigmoid / sigmoid / mse", ['sigmoid', 'sigmoid'], 'mse'),
    ("sigmoid / softmax / mse", ['sigmoid', 'softmax'], 'mse'),
    ("sigmoid / softmax / CE", ['sigmoid', 'softmax'], 'cross_entropy'),
    ("tanh / softmax / CE", ['tanh', 'softmax'], 'cross_entropy'),
    ("relu / softmax / CE", ['relu', 'softmax'], 'cross_entropy'),
]

print("Evolution de la loss moyenne (train) :")
results_multi = []
for name, acts, loss in configs_multi:
    network, history = run_config(b_train, b_test, 6, acts, loss, 3, 0.05, 100)
    show_history(name, history)
    results_multi.append((name, accuracy(network, b_train, acts), accuracy(network, b_test, acts)))

print("\nConfiguration                train acc   test acc")
for name, tr, te in results_multi:
    print(f"{name:<28} {tr:.3f}       {te:.3f}")

Evolution de la loss moyenne (train) :


sigmoid / sigmoid / mse     ep0=1.030  ep19=0.233  ep39=0.167  ep59=0.151  ep79=0.144  ep99=0.140


sigmoid / softmax / mse     ep0=0.683  ep19=0.153  ep39=0.135  ep59=0.131  ep79=0.129  ep99=0.127


sigmoid / softmax / CE      ep0=1.040  ep19=0.233  ep39=0.227  ep59=0.225  ep79=0.224  ep99=0.224


tanh / softmax / CE         ep0=0.573  ep19=0.244  ep39=0.239  ep59=0.230  ep79=0.223  ep99=0.216


relu / softmax / CE         ep0=0.654  ep19=0.257  ep39=0.256  ep59=0.256  ep79=0.261  ep99=0.261

Configuration                train acc   test acc
sigmoid / sigmoid / mse      0.910       0.889
sigmoid / softmax / mse      0.910       0.878
sigmoid / softmax / CE       0.910       0.889
tanh / softmax / CE          0.919       0.878
relu / softmax / CE          0.871       0.844


Ici toutes les configurations sont proches (entre 0.84 et 0.89 sur le test, baseline à 0.278). L'accuracy train plafonne aussi autour de 0.91 : les nuages se chevauchent, donc une partie des points est mal classée quel que soit le réseau. Le choix activation / loss compte peu sur ce problème.

La relu est un peu en dessous et sa loss remonte légèrement à la fin, ce qui ressemble à un learning rate un peu trop grand pour elle ou à des neurones relu bloqués à 0.

#### Problème 3 : régression

y = 3 x1 - 2 x2 + 0.5 + un petit bruit gaussien. La sortie est forcément linéaire (une sigmoïde ne peut sortir que entre 0 et 1) et la loss est la mse. On compare l'activation de la couche cachée. On mesure l'erreur avec la RMSE sur le test.

In [21]:
def make_regression(n, seed):
    random.seed(seed)
    data = []
    for _ in range(n):
        x1, x2 = random.uniform(-1, 1), random.uniform(-1, 1)
        data.append([x1, x2, 3 * x1 - 2 * x2 + 0.5 + random.gauss(0, 0.1)])
    return data


def rmse(network, data, activations):
    return math.sqrt(sum((predict_g(network, r, activations, 'regression') - r[-1])**2 for r in data) / len(data))


reg = make_regression(300, seed=3)
r_train, r_test = train_test_split(reg)
standardize(r_train, r_test)

mean_y = sum(r[-1] for r in r_train) / len(r_train)
print("Baseline moyenne de y, RMSE test :", round(math.sqrt(sum((mean_y - r[-1])**2 for r in r_test) / len(r_test)), 3))

Baseline moyenne de y, RMSE test : 2.08


In [22]:
configs_reg = [
    ("linear / linear", ['linear', 'linear']),
    ("sigmoid / linear", ['sigmoid', 'linear']),
    ("tanh / linear", ['tanh', 'linear']),
    ("relu / linear", ['relu', 'linear']),
]

print("Evolution de la loss moyenne (train) :")
results_reg = []
for name, acts in configs_reg:
    network, history = run_config(r_train, r_test, 4, acts, 'mse', 1, 0.01, 100, task='regression')
    show_history(name, history)
    results_reg.append((name, network, acts, rmse(network, r_train, acts), rmse(network, r_test, acts)))

print("\nConfiguration       RMSE train   RMSE test")
for name, _, _, tr, te in results_reg:
    print(f"{name:<20} {tr:.3f}        {te:.3f}")

Evolution de la loss moyenne (train) :
linear / linear             ep0=1.834  ep19=0.011  ep39=0.011  ep59=0.011  ep79=0.011  ep99=0.011


sigmoid / linear            ep0=3.848  ep19=0.062  ep39=0.041  ep59=0.033  ep79=0.028  ep99=0.025


tanh / linear               ep0=2.738  ep19=0.022  ep39=0.015  ep59=0.013  ep79=0.013  ep99=0.013
relu / linear               ep0=2.597  ep19=0.013  ep39=0.012  ep59=0.012  ep79=0.012  ep99=0.012

Configuration       RMSE train   RMSE test
linear / linear      0.107        0.095
sigmoid / linear     0.155        0.171
tanh / linear        0.113        0.113
relu / linear        0.106        0.101


In [23]:
name, network, acts, _, _ = results_reg[0]
print(f"Quelques prédictions sur le test ({name}) :")
for r in r_test[:8]:
    print(f"vrai={r[-1]:7.3f}   prédit={predict_g(network, r, acts, 'regression'):7.3f}")

Quelques prédictions sur le test (linear / linear) :
vrai=  0.798   prédit=  0.850
vrai=  1.259   prédit=  1.223
vrai=  2.940   prédit=  3.062
vrai= -0.008   prédit= -0.031
vrai=  0.719   prédit=  0.540
vrai= -3.255   prédit= -3.209
vrai=  3.084   prédit=  2.980
vrai=  4.096   prédit=  4.092


La baseline (prédire toujours la moyenne) donne une RMSE de 2.08. Comme y est une fonction linéaire de x1 et x2, le réseau tout linéaire suffit : une couche cachée linéaire suivie d'une sortie linéaire reste une fonction linéaire. Il arrive à une RMSE de 0.095 sur le test, du même ordre que le bruit ajouté (écart-type 0.1), donc on ne peut pas vraiment faire mieux.

Relu et tanh s'en approchent. La sigmoïde est la plus lente et n'a pas fini de converger après 100 epochs.

## 12. Application au Seeds Dataset

### Question 21 : chargement

Dans le fichier fourni, chaque ligne est une seule cellule entre guillemets avec les 8 valeurs séparées par des tabulations. On lit donc avec `csv.reader` puis on découpe sur les espaces. Les classes vont de 1 à 3 dans le fichier, on les ramène à 0, 1, 2.

Sur Colab il faut d'abord uploader le fichier (`from google.colab import files` puis `files.upload()`).

In [24]:
def load_csv(filename):
    dataset = []
    with open(filename, 'r') as file:
        for row in csv.reader(file):
            if not row:
                continue
            dataset.append(" ".join(row).split())
    return dataset


def convert_dataset(dataset):
    for row in dataset:
        for i in range(len(row) - 1):
            row[i] = float(row[i])
    labels = sorted(set(row[-1] for row in dataset))
    lookup = {label: i for i, label in enumerate(labels)}
    for row in dataset:
        row[-1] = lookup[row[-1]]
    return lookup


seeds = load_csv('seeds_dataset.csv')
lookup = convert_dataset(seeds)
print("Nombre de lignes :", len(seeds))
print("Correspondance des classes :", lookup)
print("Première ligne :", seeds[0])

Nombre de lignes : 210
Correspondance des classes : {'1': 0, '2': 1, '3': 2}
Première ligne : [15.26, 14.84, 0.871, 5.763, 3.312, 2.221, 5.22, 0]


### Question 22 : taille du réseau

In [25]:
n_inputs = len(seeds[0]) - 1
n_outputs = len(set(row[-1] for row in seeds))
print("n_inputs =", n_inputs, " n_outputs =", n_outputs)
print("Exemples par classe :", [sum(r[-1] == c for r in seeds) for c in range(n_outputs)])

n_inputs = 7  n_outputs = 3
Exemples par classe : [70, 70, 70]


### Question 23 : entraînement et prédiction

Les features n'ont pas du tout la même échelle (la surface autour de 15, la compacité autour de 0.87), donc on standardise avec les stats du train. On garde 30 % des données en test pour voir si le réseau généralise, sinon afficher les prédictions sur les données d'entraînement ne prouve pas grand chose. On utilise le réseau de base du TD (sigmoïde + erreur quadratique) avec 10 neurones cachés.

In [26]:
s_train, s_test = train_test_split(seeds, ratio=0.7, seed=1)
standardize(s_train, s_test)
print("train :", len(s_train), " test :", len(s_test))
print("Baseline classe majoritaire (test) :", round(majority_baseline(s_train, s_test), 3))
print()

random.seed(1)
seeds_net = initialize_network(n_inputs, 10, n_outputs)
train_network(seeds_net, s_train, 0.1, 300, n_outputs, print_every=30)

train : 147  test : 63
Baseline classe majoritaire (test) : 0.317

epoch=0, lrate=0.1, error=227.454
epoch=30, lrate=0.1, error=15.808


epoch=60, lrate=0.1, error=10.421
epoch=90, lrate=0.1, error=8.471


epoch=120, lrate=0.1, error=7.404
epoch=150, lrate=0.1, error=6.406


epoch=180, lrate=0.1, error=5.478
epoch=210, lrate=0.1, error=4.781


epoch=240, lrate=0.1, error=4.096
epoch=270, lrate=0.1, error=3.519


epoch=299, lrate=0.1, error=3.073


In [27]:
def evaluate(network, data):
    return sum(predict(network, r) == r[-1] for r in data) / len(data)


print("Prédictions sur le test :")
for row in s_test:
    p = predict(seeds_net, row)
    print(f"Attendu={row[-1]}, Prédit={p}" + ("" if p == row[-1] else "   <- erreur"))

print(f"\nAccuracy train : {evaluate(seeds_net, s_train):.3f}")
print(f"Accuracy test  : {evaluate(seeds_net, s_test):.3f}")

Prédictions sur le test :
Attendu=1, Prédit=1
Attendu=0, Prédit=0
Attendu=1, Prédit=1
Attendu=2, Prédit=2
Attendu=1, Prédit=1
Attendu=1, Prédit=1
Attendu=2, Prédit=2
Attendu=1, Prédit=1
Attendu=0, Prédit=0
Attendu=0, Prédit=0
Attendu=2, Prédit=2
Attendu=1, Prédit=1
Attendu=2, Prédit=0   <- erreur
Attendu=1, Prédit=1
Attendu=1, Prédit=1
Attendu=2, Prédit=2
Attendu=2, Prédit=2
Attendu=1, Prédit=1
Attendu=0, Prédit=0
Attendu=2, Prédit=2
Attendu=2, Prédit=2
Attendu=1, Prédit=1
Attendu=0, Prédit=0
Attendu=1, Prédit=0   <- erreur
Attendu=2, Prédit=2
Attendu=1, Prédit=1
Attendu=0, Prédit=0
Attendu=2, Prédit=0   <- erreur
Attendu=0, Prédit=0
Attendu=1, Prédit=0   <- erreur
Attendu=2, Prédit=2
Attendu=0, Prédit=0
Attendu=0, Prédit=0
Attendu=2, Prédit=2
Attendu=1, Prédit=1
Attendu=0, Prédit=0
Attendu=2, Prédit=2
Attendu=0, Prédit=0
Attendu=2, Prédit=2
Attendu=0, Prédit=0
Attendu=1, Prédit=1
Attendu=2, Prédit=2
Attendu=0, Prédit=0
Attendu=2, Prédit=2
Attendu=1, Prédit=1
Attendu=1, Prédit=1
Attend

In [28]:
confusion = [[0] * n_outputs for _ in range(n_outputs)]
for row in s_test:
    confusion[row[-1]][predict(seeds_net, row)] += 1

print("Matrice de confusion (lignes = vraie classe, colonnes = prédite)")
for c in range(n_outputs):
    print(c, confusion[c])

Matrice de confusion (lignes = vraie classe, colonnes = prédite)
0 [20, 0, 0]
1 [3, 19, 0]
2 [3, 0, 18]


Le réseau fait 0.905 sur le test contre 0.317 pour la baseline.

Toutes les erreurs sont des classes 1 et 2 prédites en 0. Ça se comprend : la classe 0 (Kama) a des grains de taille intermédiaire entre la classe 1 (Rosa, les plus gros) et la classe 2 (Canadian, les plus petits), donc c'est elle qui est à la frontière des deux autres.

On a 1.000 en train contre 0.905 en test, le réseau commence à sur-apprendre. Avec seulement 63 exemples de test, un point change l'accuracy de 1.6 %, donc une validation croisée donnerait un chiffre plus fiable.

## 13. Bonus : plusieurs couches cachées

### Question 24

Au lieu d'un entier, `n_hidden` est une liste avec le nombre de neurones de chaque couche cachée. Le code est le même qu'à la question 1, seule la liste des tailles change.

In [29]:
def initialize_network_multiple_hidden(n_inputs, n_hidden, n_outputs):
    network = []
    variable = [n_inputs] + list(n_hidden) + [n_outputs]
    for i in range(1, len(variable)):
        layer = []
        for j in range(variable[i]):
            neuro = {"weights": [round(random.random(), 1) for _ in range(variable[i-1] + 1)]}
            layer.append(neuro)
        network.append(layer)
    return network


random.seed(1)
n1 = initialize_network(2, 3, 4)
random.seed(1)
n2 = initialize_network_multiple_hidden(2, [3], 4)
print("Avec [3] on retrouve la question 1 :", n1 == n2)

random.seed(1)
deep = initialize_network_multiple_hidden(2, [3, 5, 4], 2)
print("Nombre de couches :", len(deep), "| neurones par couche :", [len(l) for l in deep],
      "| poids par neurone :", [len(l[0]['weights']) for l in deep])

Avec [3] on retrouve la question 1 : True
Nombre de couches : 4 | neurones par couche : [3, 5, 4, 2] | poids par neurone : [3, 4, 6, 5]


### Question 25

`forward_propagate` et `backward_propagate_error` marchent sans modification parce qu'elles ne supposent jamais qu'il y a exactement 2 couches :

- elles bouclent sur `range(len(network))`, donc sur toutes les couches quel que soit leur nombre ;
- elles ne regardent que les couches voisines avec des indices relatifs : `network[i-1]` pour récupérer les entrées en avant, `network[i+1]` pour récupérer les deltas en arrière ;
- les seuls cas particuliers sont la première couche (qui reçoit `row`) et la dernière (qui reçoit `expected`), et ils sont repérés avec `i == 0` et `i == len(network) - 1`, pas avec un numéro de couche fixe.

`update_weights` a la même structure, donc elle marche aussi. On le vérifie en entraînant des réseaux plus profonds sur le Seeds Dataset avec les fonctions de base, sans rien changer.

In [30]:
for hidden in ([10], [8, 5], [3, 5, 4]):
    random.seed(1)
    net_h = initialize_network_multiple_hidden(n_inputs, hidden, n_outputs)
    print("Couches cachées", hidden)
    train_network(net_h, s_train, 0.1, 300, n_outputs, print_every=100)
    print(f"accuracy train={evaluate(net_h, s_train):.3f}  test={evaluate(net_h, s_test):.3f}\n")

Couches cachées [10]
epoch=0, lrate=0.1, error=227.454


epoch=100, lrate=0.1, error=8.060


epoch=200, lrate=0.1, error=5.012


epoch=299, lrate=0.1, error=3.073
accuracy train=1.000  test=0.905

Couches cachées [8, 5]
epoch=0, lrate=0.1, error=169.946


epoch=100, lrate=0.1, error=8.686


epoch=200, lrate=0.1, error=5.314


epoch=299, lrate=0.1, error=4.528
accuracy train=0.986  test=0.873

Couches cachées [3, 5, 4]
epoch=0, lrate=0.1, error=174.615


epoch=100, lrate=0.1, error=34.573


epoch=200, lrate=0.1, error=5.675


epoch=299, lrate=0.1, error=4.324
accuracy train=0.986  test=0.873



Les réseaux à plusieurs couches s'entraînent avec les fonctions de base sans modification, ce qui confirme la question 25.

Ils ne font pas mieux que la couche unique de 10 neurones : le dataset est petit et presque séparable, ajouter des couches n'apporte rien ici. Le réseau [3, 5, 4] démarre plus lentement (erreur encore à 34.6 à l'epoch 100). C'est l'effet des sigmoïdes empilées : à chaque couche le delta est multiplié par une dérivée d'au plus 0.25, donc il devient petit dans les premières couches.